# Building upon the model and evaluate method used in InsectNet

https://academic.oup.com/pnasnexus/article/4/1/pgae575/7933354?login=false

## Overview

Thoughts: The code will have to be updated in the sense that it is not made at all to handle any large batches of images, but the general gist of the preprocessing could be something to work with.

Not sure if all flowers in the image should be analyzed or if just one (there is a green marker on one of them for example). So right now it is detecting the marker and grabbing the flowers nearby, or an option to choose the areay yourself when you run with the argument --roi. 

Argument --debug save images that give some visual information such as the zone used, difference, countours and if a possible match: crop. Make sure you have folder debug/ made if you try this.

So it goes something like this right now:
- Defining the watching zone, either by auto-detect a green marker and use a circle around it, or manually draw a rectangle (using --roi)
- Building median background (this has to be reworked when working with larger batches, as in instead of loading all images it has to load a very small subset, perhaps 8)
- Per-frame detection: finding pixels darker than the median, exclude green vegetation, then filter contours by size, shape, and
texture
- Removing static detections: rejecting positions that appear in too many frames (soil/shadows, not insects). This could be too specific to this solution, right now it is set that if something comes up in two frames around the same area, it is disregarded
- Classification: crop each detection and run through the RegNet insect classifier, this is using the model and evaluation code


### Imports and constants

In [1]:
import sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torchvision

from InsectNet.evaluate import evaluate # import from InsectNet

In [2]:
PROJECT_DIR = Path.cwd() # adjusting to jupyter notebooks

MIN_CONTOUR_AREA = 800  # minimum pixel area to count as a visitor
DARKER_THRESHOLD = 40  # how much darker than median to count as foreground
PADDING = 50  # extra pixels around the detected region
MAX_ASPECT_RATIO = 3  # reject elongated detections (stems/grass)
MIN_TEXTURE = 50  # minimum grayscale std dev — insects have texture, soil doesn't
STATIC_DIST = 80  # pixels — detections closer than this are "same position"
STATIC_MAX_FRAMES = 2  # reject detections appearing in more than this many frames

# Marker detection (vivid green/turquoise clip)
MARKER_HUE = (45, 75)
MARKER_SAT_MIN = 200
MARKER_VAL_MIN = 100
MARKER_MIN_AREA = 200
MARKER_ZONE_RADIUS = 800  # fixed radius around marker

### Helper methods

In [3]:
def load_model():
    weights = torch.load(
        PROJECT_DIR / "InsectNet" / "model.pth",
        map_location=torch.device("cpu"),
        weights_only=False,
    )["model"]

    model = torchvision.models.regnet_y_32gf()
    model.fc = torch.nn.Linear(3712, 2526)
    model.load_state_dict(weights, strict=True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    model.eval()
    return model


def find_marker(image):
    """Find the green marker clip. Returns centroid (cx, cy) or None."""
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(
        hsv,
        (MARKER_HUE[0], MARKER_SAT_MIN, MARKER_VAL_MIN),
        (MARKER_HUE[1], 255, 255),
    )

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    clusters = [c for c in contours if cv2.contourArea(c) > MARKER_MIN_AREA]
    if not clusters:
        return None

    h_img, w_img = image.shape[:2]
    cx_img, cy_img = w_img // 2, h_img // 2

    def dist_to_center(c):
        M = cv2.moments(c)
        if M["m00"] == 0:
            return float("inf")
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
        return (cx - cx_img) ** 2 + (cy - cy_img) ** 2

    best = min(clusters, key=dist_to_center)
    M = cv2.moments(best)
    return int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])


def build_marker_zone(image, marker):
    """Create a circular zone around the marker."""
    h_img, w_img = image.shape[:2]
    zone = np.zeros((h_img, w_img), dtype=np.uint8)
    cv2.circle(zone, marker, MARKER_ZONE_RADIUS, 255, -1)
    return zone


def select_roi(image_path):
    """Open the first image and let user draw a rectangle as the watching zone."""
    img = cv2.imread(image_path)
    h, w = img.shape[:2]
    scale = min(1.0, 1200 / w)
    display = cv2.resize(img, (int(w * scale), int(h * scale)))

    print("Draw a rectangle around the flower area, then press ENTER or SPACE.")
    roi = cv2.selectROI("Select flower region", display, showCrosshair=True)
    cv2.destroyAllWindows()

    x, y, rw, rh = roi
    x, y, rw, rh = int(x / scale), int(y / scale), int(rw / scale), int(rh / scale)

    if rw == 0 or rh == 0:
        return None

    zone = np.zeros((h, w), dtype=np.uint8)
    zone[y : y + rh, x : x + rw] = 255
    print(f"ROI selected: x={x} y={y} w={rw} h={rh}")
    return zone


def detect_visitor(image, background, zone):
    """
    Detect dark objects (insects) that appear in the watching zone but not in
    the median background. Returns list of bounding boxes [(x, y, w, h), ...].

    Strategy:
    1. Find pixels that are DARKER than the median — insects are dark-bodied
    2. Exclude green vegetation pixels — universal for outdoor scenes
    3. Restrict to watching zone
    4. Filter by area, aspect ratio, and texture
    """
    # Find where current frame is darker than median
    gray_img = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY).astype(np.int16)
    gray_bg = cv2.cvtColor(background, cv2.COLOR_BGR2GRAY).astype(np.int16)
    darker = np.clip(gray_bg - gray_img, 0, 255).astype(np.uint8)

    darker = cv2.GaussianBlur(darker, (7, 7), 0)
    _, mask = cv2.threshold(darker, DARKER_THRESHOLD, 255, cv2.THRESH_BINARY)

    # Restrict to watching zone
    mask = cv2.bitwise_and(mask, zone)

    # Exclude green vegetation — universal for outdoor scenes
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    green_pixels = cv2.inRange(hsv, (25, 40, 40), (95, 255, 255))
    mask = cv2.bitwise_and(mask, cv2.bitwise_not(green_pixels))

    # Light morphology — small open to remove speckles, small close to fill gaps
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return []

    valid = []
    for c in contours:
        area = cv2.contourArea(c)
        if area < MIN_CONTOUR_AREA:
            continue

        x, y, w, h = cv2.boundingRect(c)

        aspect = max(w, h) / max(min(w, h), 1)
        if aspect > MAX_ASPECT_RATIO:
            continue

        # Reject low-texture regions (soil, shadow) — insects have visible detail
        roi = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)[y : y + h, x : x + w]
        if roi.std() < MIN_TEXTURE:
            continue

        valid.append((area, cv2.boundingRect(c)))

    if not valid:
        return []

    valid.sort(key=lambda x: x[0], reverse=True)
    return [bbox for _, bbox in valid]


def crop_with_padding(image, bbox):
    """Crop image to bounding box with padding."""
    x, y, w, h = bbox
    h_img, w_img = image.shape[:2]
    x1 = max(0, x - PADDING)
    y1 = max(0, y - PADDING)
    x2 = min(w_img, x + w + PADDING)
    y2 = min(h_img, y + h + PADDING)
    return image[y1:y2, x1:x2]


def save_debug_images(name, image, background, zone, marker, debug_dir):
    """Save debug images for visual inspection."""
    h_img, w_img = image.shape[:2]

    # 1. Watching zone overlay with marker
    overlay = image.copy()
    overlay[zone > 0] = overlay[zone > 0] // 2 + np.array(
        [128, 0, 128], dtype=np.uint8
    )
    if marker is not None:
        cv2.circle(overlay, marker, 15, (0, 255, 0), -1)
    cv2.imwrite(str(debug_dir / f"{name}_1_zone.jpg"), overlay)

    # 2. Darker-than-median mask (after vegetation exclusion + morphology)
    gray_img = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY).astype(np.int16)
    gray_bg = cv2.cvtColor(background, cv2.COLOR_BGR2GRAY).astype(np.int16)
    darker = np.clip(gray_bg - gray_img, 0, 255).astype(np.uint8)
    darker = cv2.GaussianBlur(darker, (7, 7), 0)
    _, mask = cv2.threshold(darker, DARKER_THRESHOLD, 255, cv2.THRESH_BINARY)
    mask = cv2.bitwise_and(mask, zone)

    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    green_pixels = cv2.inRange(hsv, (25, 40, 40), (95, 255, 255))
    mask = cv2.bitwise_and(mask, cv2.bitwise_not(green_pixels))

    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)
    cv2.imwrite(str(debug_dir / f"{name}_2_diff.jpg"), mask)

    # 3. Contours (green=pass, red=rejected, skip tiny ones)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    overlay2 = image.copy()
    for c in contours:
        area = cv2.contourArea(c)
        if area < 100:
            continue
        x, y, w, h = cv2.boundingRect(c)
        aspect = max(w, h) / max(min(w, h), 1)

        roi = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)[y : y + h, x : x + w]
        tex = roi.std()
        passes = (
            area >= MIN_CONTOUR_AREA
            and aspect <= MAX_ASPECT_RATIO
            and tex >= MIN_TEXTURE
        )
        color = (0, 255, 0) if passes else (0, 0, 255)
        cv2.drawContours(overlay2, [c], -1, color, 2)
        cv2.rectangle(overlay2, (x, y), (x + w, y + h), color, 2)
        label = f"a={int(area)} t={tex:.0f}"
        cv2.putText(
            overlay2, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1
        )

    cv2.imwrite(str(debug_dir / f"{name}_3_contours.jpg"), overlay2)


def filter_static_detections(all_detections):
    """
    Remove detections that appear at the same position across too many frames.
    A real visitor shows up in a few frames then leaves. A soil patch or shadow
    persists across many frames.

    Args:
        all_detections: dict of {path: [(x, y, w, h), ...]}

    Returns:
        dict with static detections removed
    """
    # Collect all detection centers across all frames
    all_centers = []  # (cx, cy, path)
    for path, bboxes in all_detections.items():
        for x, y, w, h in bboxes:
            all_centers.append((x + w // 2, y + h // 2, path))

    # For each detection, count how many different frames have a detection nearby
    static_positions = set()  # (cx, cy) positions that appear too often
    for cx, cy, _ in all_centers:
        frames_with_nearby = set()
        for cx2, cy2, path2 in all_centers:
            dist = ((cx - cx2) ** 2 + (cy - cy2) ** 2) ** 0.5
            if dist < STATIC_DIST:
                frames_with_nearby.add(path2)
        if len(frames_with_nearby) > STATIC_MAX_FRAMES:
            static_positions.add((cx, cy))

    if static_positions:
        n = len(static_positions)
        print(
            f"Filtered {n} static position(s) (appeared in >{STATIC_MAX_FRAMES} frames)"
        )

    # Remove detections near static positions
    filtered = {}
    for path, bboxes in all_detections.items():
        kept = []
        for x, y, w, h in bboxes:
            cx, cy = x + w // 2, y + h // 2
            is_static = any(
                ((cx - sx) ** 2 + (cy - sy) ** 2) ** 0.5 < STATIC_DIST
                for sx, sy in static_positions
            )
            if not is_static:
                kept.append((x, y, w, h))
        filtered[path] = kept

    return filtered

## Main method

In [10]:
def main(image_paths, debug=False, manual_roi=False):
    paths = image_paths
    debug_dir = PROJECT_DIR / "data" / "debug"

    if debug:
        debug_dir.mkdir(exist_ok=True)
        print(f"Debug images will be saved to {debug_dir}/")

    first_img = cv2.imread(paths[0])
    marker = None

    if manual_roi:
        zone = select_roi(paths[0])
        if zone is None:
            print("No region selected.")
            sys.exit(1)
    else:
        marker = find_marker(first_img)
        if marker is None:
            print("No green marker found. Use --roi to manually select a region.")
            sys.exit(1)
        print(f"Marker found at ({marker[0]}, {marker[1]})")
        zone = build_marker_zone(first_img, marker)

    pct = 100 * np.count_nonzero(zone) / zone.size
    print(f"Watching zone covers {pct:.1f}% of image")

    print("Computing median background...")
    frames = []
    for p in paths:
        img = cv2.imread(p)
        if img is not None:
            frames.append(img)
    background = np.median(frames, axis=0).astype(np.uint8)

    print("Detecting visitors...")
    images = {}
    all_detections = {}
    for path in paths:
        image = cv2.imread(path)
        if image is None:
            continue
        images[path] = image
        name = Path(path).stem

        if debug:
            save_debug_images(name, image, background, zone, marker, debug_dir)

        all_detections[path] = detect_visitor(image, background, zone)

    filtered = filter_static_detections(all_detections)

    print("Loading model...")
    model = load_model()
    cmn_df = pd.read_csv(PROJECT_DIR / "InsectNet" / "classes.csv")
    class_txt_path = str(PROJECT_DIR / "InsectNet" / "classes.txt")
    print("Model loaded.\n")

    for path in paths:
        if path not in images:
            continue

        image = images[path]
        name = Path(path).stem
        detections = filtered.get(path, [])

        if not detections:
            print(f"{path}: no visitor detected")
            continue

        for i, bbox in enumerate(detections):
            crop = crop_with_padding(image, bbox)
            crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

            if debug:
                cv2.imwrite(str(debug_dir / f"{name}_4_crop_{i}.jpg"), crop)

            sci, cmn, order, family, role, confirmed, other, confidence, elapsed = evaluate(
                model, crop_rgb, cmn_df, class_txt_path
            )

            x, y, w, h = bbox
            header = f"Image:           {path}"
            if len(detections) > 1:
                header += f"  (visitor {i + 1}/{len(detections)})"
            print(header)
            print(f"Region:          x={x} y={y} w={w} h={h}")
            print(f"Scientific Name: {sci}")
            print(f"Common Name:     {cmn}")
            print(f"Order:           {order}")
            print(f"Family:          {family}")
            print(f"Role:            {role}")
            print(f"Confirmed:       {confirmed}")
            if other:
                print(f"Other plausible: {other}")
            print()

In [11]:
# Som workarounds when running it in Jupyterlab compared to locally
from glob import glob

paths = sorted(glob("/workspace/Engineering Thesis/Pollinator/data/images/*.JPG"))
main(paths, debug=True) # run as is, do note use --roi flag unless running locally and not in jyputerlab container

Debug images will be saved to /workspace/Engineering Thesis/Pollinator/data/debug/
Marker found at (1294, 1344)
Watching zone covers 30.3% of image
Computing median background...
Detecting visitors...
Filtered 3 static position(s) (appeared in >2 frames)
Loading model...
Model loaded.

/workspace/Engineering Thesis/Pollinator/data/images/WSCT2946.JPG: no visitor detected
Image:           /workspace/Engineering Thesis/Pollinator/data/images/WSCT2947.JPG
Region:          x=903 y=1189 w=32 h=74
Scientific Name: Camponotus niveosetosus
Common Name:      hairy sugar ant
Order:           Hymenoptera
Family:          Formicidae
Role:            Predator
Confirmed:       False

/workspace/Engineering Thesis/Pollinator/data/images/WSCT2948.JPG: no visitor detected
/workspace/Engineering Thesis/Pollinator/data/images/WSCT2949.JPG: no visitor detected
/workspace/Engineering Thesis/Pollinator/data/images/WSCT2950.JPG: no visitor detected
/workspace/Engineering Thesis/Pollinator/data/images/WSCT295